In [2]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from ultralytics import YOLO


def find_data_yaml():
    current_folder = Path.cwd().resolve()

    search_roots = [
        current_folder,
        current_folder.parent,
        current_folder.parent.parent
    ]

    for root in search_roots:
        expected_path = (
            root
            / "smartvision_dataset"
            / "detection"
            / "data.yaml"
        )

        if expected_path.exists():
            return expected_path.resolve()

    for root in search_roots:
        matches = list(root.rglob("data.yaml"))

        for match in matches:
            if (
                match.parent.name == "detection"
                and match.parent.parent.name == "smartvision_dataset"
            ):
                return match.resolve()

    raise FileNotFoundError(
        "data.yaml was not found. "
        "Place it inside "
        "smartvision_dataset/detection/data.yaml"
    )


PROJECT_DIR = Path.cwd().resolve()

DATA_YAML = find_data_yaml()
DETECTION_DIR = DATA_YAML.parent
IMAGES_DIR = DETECTION_DIR / "images"
LABELS_DIR = DETECTION_DIR / "labels"

MODEL_WEIGHTS = "yolov8n.pt"
RUN_NAME = "smartvision_yolo"

RUNS_DIR = PROJECT_DIR / "runs" / "detect"
RUN_DIR = RUNS_DIR / RUN_NAME
OUTPUT_DIR = PROJECT_DIR / "outputs" / "yolo_results"

EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 16

DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = 0

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
def verify_dataset():
    print("\nVERIFYING DATASET")
    print("=" * 60)

    if not DATA_YAML.exists():
        raise FileNotFoundError(
            f"Dataset YAML not found: {DATA_YAML}"
        )

    if not IMAGES_DIR.exists():
        raise FileNotFoundError(
            f"Images folder not found: {IMAGES_DIR}"
        )

    if not LABELS_DIR.exists():
        raise FileNotFoundError(
            f"Labels folder not found: {LABELS_DIR}"
        )

    with DATA_YAML.open("r", encoding="utf-8") as file:
        data_config = yaml.safe_load(file)

    if "train" not in data_config:
        raise KeyError("The data.yaml file must contain a 'train' path.")

    if "val" not in data_config:
        raise KeyError("The data.yaml file must contain a 'val' path.")

    names = data_config.get("names", [])

    if isinstance(names, dict):
        class_names = [
            names[key]
            for key in sorted(
                names,
                key=lambda value: int(value)
            )
        ]
    else:
        class_names = list(names)

    image_extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp"
    }

    image_files = [
        file
        for file in IMAGES_DIR.iterdir()
        if file.suffix.lower() in image_extensions
    ]

    label_files = list(
        LABELS_DIR.glob("*.txt")
    )

    image_stems = {
        file.stem
        for file in image_files
    }

    label_stems = {
        file.stem
        for file in label_files
    }

    missing_labels = sorted(
        image_stems - label_stems
    )

    print("Images       :", len(image_files))
    print("Label files  :", len(label_files))
    print("Classes      :", len(class_names))
    print("Class names  :", class_names)
    print("Train path   :", data_config["train"])
    print("Val path     :", data_config["val"])

    if missing_labels:
        print(
            "Images without label files:",
            len(missing_labels)
        )
    else:
        print("Every image has a matching label file.")

    return data_config

In [4]:
def train_yolo():
    print("\nTRAINING YOLOV8")
    print("=" * 60)

    model = YOLO(MODEL_WEIGHTS)

    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        patience=20,
        pretrained=True,
        optimizer="Adam",
        lr0=0.001,
        lrf=0.01,
        momentum=0.9,
        weight_decay=0.0005,
        warmup_epochs=3,
        cos_lr=True,
        box=7.5,
        cls=0.5,
        dfl=1.5,
        plots=True,
        save=True,
        verbose=True
    )

    best_weights = RUN_DIR / "weights" / "best.pt"

    if not best_weights.exists():
        raise FileNotFoundError(
            f"Best model was not found: {best_weights}"
        )

    best_model = YOLO(str(best_weights))

    print("\nTraining complete.")
    print("Best model:", best_weights)

    return best_model, results

In [5]:
def evaluate_yolo(model):
    print("\nEVALUATING YOLOV8")
    print("=" * 60)

    results = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        conf=0.25,
        iou=0.60,
        plots=True
    )

    print(f"mAP@0.5      : {results.box.map50:.4f}")
    print(f"mAP@0.5:0.95 : {results.box.map:.4f}")
    print(f"Precision    : {results.box.mp:.4f}")
    print(f"Recall       : {results.box.mr:.4f}")

    return results

In [6]:
def test_predictions(model, num_samples=6):
    print("\nTESTING SAMPLE IMAGES")
    print("=" * 60)

    image_extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp"
    }

    image_files = sorted([
        file
        for file in IMAGES_DIR.iterdir()
        if file.suffix.lower() in image_extensions
    ])[:num_samples]

    if not image_files:
        raise FileNotFoundError(
            f"No images were found in: {IMAGES_DIR}"
        )

    columns = 3
    rows = (len(image_files) + columns - 1) // columns

    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(18, 6 * rows)
    )

    if rows == 1:
        axes = axes.reshape(1, -1)

    axes = axes.flatten()

    for index, image_path in enumerate(image_files):
        results = model.predict(
            source=str(image_path),
            conf=0.25,
            iou=0.60,
            imgsz=IMG_SIZE,
            device=DEVICE,
            save=False,
            verbose=False
        )

        annotated_image = results[0].plot()
        annotated_image = cv2.cvtColor(
            annotated_image,
            cv2.COLOR_BGR2RGB
        )

        detections = len(results[0].boxes)

        axes[index].imshow(annotated_image)
        axes[index].set_title(
            f"{image_path.name}\n"
            f"{detections} objects detected"
        )
        axes[index].axis("off")

    for index in range(
        len(image_files),
        len(axes)
    ):
        axes[index].axis("off")

    output_path = OUTPUT_DIR / "yolo_predictions.png"

    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    print("Predictions saved:", output_path)

In [7]:
def visualize_training_results():
    print("\nVISUALIZING TRAINING RESULTS")
    print("=" * 60)

    results_csv = RUN_DIR / "results.csv"

    if not results_csv.exists():
        raise FileNotFoundError(
            f"Training results not found: {results_csv}"
        )

    results_df = pd.read_csv(results_csv)
    results_df.columns = (
        results_df.columns
        .str.strip()
    )

    figure, axes = plt.subplots(
        2,
        2,
        figsize=(14, 10)
    )

    axes[0, 0].plot(
        results_df["metrics/mAP50(B)"],
        label="mAP@0.5"
    )
    axes[0, 0].plot(
        results_df["metrics/mAP50-95(B)"],
        label="mAP@0.5:0.95"
    )
    axes[0, 0].set_title(
        "Mean Average Precision"
    )
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("mAP")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(
        results_df["metrics/precision(B)"],
        label="Precision"
    )
    axes[0, 1].plot(
        results_df["metrics/recall(B)"],
        label="Recall"
    )
    axes[0, 1].set_title(
        "Precision and Recall"
    )
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Score")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(
        results_df["train/box_loss"],
        label="Box Loss"
    )
    axes[1, 0].plot(
        results_df["train/cls_loss"],
        label="Class Loss"
    )
    axes[1, 0].plot(
        results_df["train/dfl_loss"],
        label="DFL Loss"
    )
    axes[1, 0].set_title(
        "Training Loss"
    )
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Loss")
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(
        results_df["val/box_loss"],
        label="Validation Box Loss"
    )
    axes[1, 1].plot(
        results_df["val/cls_loss"],
        label="Validation Class Loss"
    )
    axes[1, 1].plot(
        results_df["val/dfl_loss"],
        label="Validation DFL Loss"
    )
    axes[1, 1].set_title(
        "Validation Loss"
    )
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Loss")
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

    output_path = (
        OUTPUT_DIR /
        "yolo_training_metrics.png"
    )

    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    print("Training graph saved:", output_path)

In [8]:
def export_model(model):
    print("\nEXPORTING MODEL")
    print("=" * 60)

    model.export(
        format="torchscript",
        imgsz=IMG_SIZE,
        device=DEVICE
    )

    model.export(
        format="onnx",
        imgsz=IMG_SIZE,
        device=DEVICE
    )

    print("Model export complete.")

In [9]:
def main():
    verify_dataset()

    model, train_results = train_yolo()

    validation_results = evaluate_yolo(model)

    test_predictions(model)

    visualize_training_results()

    export_model(model)

    print("\nYOLOV8 PIPELINE COMPLETE")
    print("=" * 60)
    print(
        f"mAP@0.5      : "
        f"{validation_results.box.map50:.4f}"
    )
    print(
        f"mAP@0.5:0.95 : "
        f"{validation_results.box.map:.4f}"
    )
    print(
        f"Precision    : "
        f"{validation_results.box.mp:.4f}"
    )
    print(
        f"Recall       : "
        f"{validation_results.box.mr:.4f}"
    )
    print(
        "Best model   :",
        RUN_DIR / "weights" / "best.pt"
    )

    return model, train_results, validation_results

In [10]:
trained_model, training_results, validation_results = main()


VERIFYING DATASET
Images       : 2210
Label files  : 2210
Classes      : 26
Class names  : ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'traffic light', 'stop sign', 'bench', 'bird', 'cat', 'dog', 'horse', 'cow', 'elephant', 'bottle', 'cup', 'bowl', 'pizza', 'cake', 'chair', 'couch', 'potted plant', 'bed']
Train path   : images
Val path     : images
Every image has a matching label file.

TRAINING YOLOV8
New https://pypi.org/project/ultralytics/8.4.114 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.13.7 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\SAKTHI\Desktop\SMARTVISION\smartvision_dataset

c:\Users\SAKTHI\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/07/31 22:47:44 INFO mlflow.tracking.fluent: Experiment with name 'C:\Users\SAKTHI\Desktop\SMARTVISION\MODEL\runs\detect' does not exist. Creating a new experiment.


MLflow: logging run_id(efd38413d1ec4200ba144fccf29ff330) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to C:\Users\SAKTHI\Desktop\SMARTVISION\MODEL\runs\detect\smartvision_yolo
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      2.22G      1.905      4.024      2.226         90        640: 100% ━━━━━━━━━━━━ 33/33 2.3it/s 14.5s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 2.8it/s 6.0s0.4s
                   all        523       1930    0.00408      0.185     0.0164    0.00599

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100       2.1G      1.793      3.595      2.168        103        640: 100% ━━━━━━━━━━━━ 33/33 2

<Figure size 1800x1200 with 6 Axes>

Predictions saved: C:\Users\SAKTHI\Desktop\SMARTVISION\MODEL\outputs\yolo_results\yolo_predictions.png

VISUALIZING TRAINING RESULTS


<Figure size 1400x1000 with 4 Axes>

Training graph saved: C:\Users\SAKTHI\Desktop\SMARTVISION\MODEL\outputs\yolo_results\yolo_training_metrics.png

EXPORTING MODEL
Ultralytics 8.4.48  Python-3.13.7 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)

PyTorch: starting from 'C:\Users\SAKTHI\Desktop\SMARTVISION\MODEL\runs\detect\smartvision_yolo\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 30, 8400) (6.0 MB)

TorchScript: starting export with torch 2.9.1+cu126...
TorchScript: export success  1.3s, saved as 'C:\Users\SAKTHI\Desktop\SMARTVISION\MODEL\runs\detect\smartvision_yolo\weights\best.torchscript' (11.9 MB)

Export complete (1.4s)
Results saved to C:\Users\SAKTHI\Desktop\SMARTVISION\MODEL\runs\detect\smartvision_yolo\weights\best.torchscript
Predict:         yolo predict task=detect model=C:\Users\SAKTHI\Desktop\SMARTVISION\MODEL\runs\detect\smartvision_yolo\weights\best.torchscript imgsz=640 
Validate:        yolo val task=detect model=C:\Users\SAKTHI\Desktop\SMAR